### 📘  Quantamental Scoring Logic & Scenario-Based Filtering ###
### (Documentation for LLM-Powered Stock Screener — MS4/MS5) ###  - by   Siri

(Do not run this notebook. Use as reference)

**1. Quantamental Scoring Components**

(Extracted from Quantamental_Model_MS4_1_v2.ipynb)

Your quantamental model computes three core scores: Technical, Fundamental, and Hybrid. These are used for stock ranking and for generating the H_Score Recommendation.

**🔹 Technical_Score**

Computed as the mean percentile rank of:

[
 "return_1m", "ema_12", "ema_26",
 "macd", "macd_signal", "macd_hist",
 "RSI_14", "volatility_21d"
]


These describe short-term momentum and trend strength.

**🔹 Fundamental_Score**

Computed as the mean percentile rank of:

[
  "payoutRatio", "cashPerShare", "dividendYield",
  "revenuePerShare", "earningsYield",
  ... (all fundamental features) ...
  "roe", "roic", "debtToEquity",
  "currentRatio", "interestCoverage"
]


These measure business quality, profitability, leverage, cash flows, and value.

**🔹 Hybrid_Score**

A balanced blend:

Hybrid_Score = 0.5 * Technical_Score + 0.5 * Fundamental_Score

**🔹 Hybrid_CS_Pct**

The cross-sectional percentile of Hybrid_Score for each month:

Hybrid_CS_Pct = percentile_rank(Hybrid_Score within month)


Used as a signal of relative strength.

**2. H_Score Recommendation Logic**

(Defined in classify_stock_v2)

Based on Hybrid_CS_Pct and the balance of Technical vs Fundamental strength, each stock receives one of five possible labels.

🟦 If Hybrid_CS_Pct ≥ 0.70 (Top 30%)

High conviction zone

✔ Short-Term Buy (Momentum)

Triggered if:

Technical_Score - Fundamental_Score > 0.05
return_1m > 0.02
RSI_14 > 50
macd_hist > 0

✔ Long-Term Buy (Fundamental)
Technical_Score - Fundamental_Score < -0.05

✔ Balanced Buy / Hold

If neither condition above is met.

🟥 If Hybrid_CS_Pct ≤ 0.30 (Bottom 30%)
→ "Avoid / Bearish"

🟨 Otherwise (Middle 40%)
→ "Hold / Neutral"

**3. Final 5 H_Score Categories**
Short-Term Buy (Momentum)
Long-Term Buy (Fundamental)
Balanced Buy / Hold
Hold / Neutral
Avoid / Bearish


These serve as high-level “signals” for user-facing recommendations.

## Mapping H_Score to Risk & Horizon ##

**4.1 Horizon Mapping**
Long-Term Oriented Labels

Long-Term Buy (Fundamental)

Balanced Buy / Hold

Hold / Neutral (only if supported by strong backtest metrics)

Short-Term Oriented Labels

Short-Term Buy (Momentum)

**4.2 Risk Mapping**

(Combines H_Score + backtest risk metrics)

✔ Low-Risk

A stock is considered low risk if:

H_Score in ["Long-Term Buy (Fundamental)", "Balanced Buy / Hold"]
vol_1m <= 0.075
max_drawdown >= -0.30
sharpe_1m_annual >= 0.8

✔ High-Risk

A stock is considered high risk if:

H_Score == "Short-Term Buy (Momentum)"
OR vol_1m >= 0.10
OR max_drawdown <= -0.35
OR sharpe_1m_annual < 0.4

❌ Always Exclude
"Avoid / Bearish"

**🟩 Scenario 1 — Long-Term + Low-Risk (LT–LR)**
**Use Case**

    * Retirement

    * Conservative profile

    * Slow & steady growth

**Filter Rules**

H_Score in ["Long-Term Buy (Fundamental)", "Balanced Buy / Hold"]

cagr >= 0.17

sharpe_1m_annual >= 0.8

vol_1m <= 0.075

max_drawdown >= -0.30

hit_rate_pos >= 0.55

**Ranking**
Sort by:

  cagr (desc)

  sharpe_1m_annual (desc)
  
  vol_1m (asc)

**Scenario 2 — Long-Term + High-Risk (LT–HR)**
**Use Case**

* Growth investors

* Multi-year compounders

* High volatility tolerance

**Filter Rules**

H_Score in [
  "Long-Term Buy (Fundamental)",
  "Balanced Buy / Hold",
  "Short-Term Buy (Momentum)"
]

cagr >= 0.24

sharpe_1m_annual >= 0.5

max_drawdown >= -0.40

Hybrid_CS_Pct >= 0.70


**Ranking**
Sort by:
  cagr (desc)
  Hybrid_Score (desc)
  vol_1m (asc)

**🟡 Scenario 3 — Short-Term + Low-Risk (ST–LR)**
Use Case

Swing traders who avoid large drawdowns

Short-term but stable setups

**Filter Rules**

H_Score in ["Short-Term Buy (Momentum)", "Balanced Buy / Hold"]

avg_fwd_1m_ret >= 0.016

vol_1m <= 0.089

max_drawdown >= -0.30

hit_rate_pos >= 0.60

**Ranking**
Sort by:

  avg_fwd_1m_ret (desc)

  hit_rate_pos (desc)
  
  vol_1m (asc)

**🔴 Scenario 4 — Short-Term + High-Risk (ST–HR)**
**Use Case**

  * High volatility traders

  * Momentum chasers

  * Short-term aggressive setups

**Filter Rules**

H_Score == "Short-Term Buy (Momentum)"

avg_fwd_1m_ret >= 0.021

Hybrid_CS_Pct >= 0.80

vol_1m >= 0.075

max_drawdown >= -0.45

sharpe_1m_annual > 0.0

**Ranking**

Sort by:

  avg_fwd_1m_ret (desc)

  Hybrid_CS_Pct (desc)
  
  vol_1m (desc)

**6. Sector Filter**

Applied after scenario selection:

if sector is not None:
    filtered = filtered[filtered["sector"] == sector]


Supports:

Single sector

Multiple sectors

**7. Default Rule (No User Input Provided)**

If user provides:

    * No risk preference

    * No time horizon

    * No sector

Then use a balanced, conservative long-term rule.

**Default Filter**
H_Score in ["Long-Term Buy (Fundamental)", "Balanced Buy / Hold"]

Hybrid_CS_Pct >= 0.80

cagr >= 0.17

sharpe_1m_annual >= 0.8

avg_fwd_1m_ret > 0

hit_rate_pos >= 0.55

max_drawdown >= -0.30

**Ranking**

Sort by Hybrid_Score (desc)
Return top 10

In [ ]:



## `filter_stocks_by_profile()` implementation

Drop this into something like:  
`src/quant_pipeline/utils/filtering.py` or `src/backend/services/filter_service.py`

```python
from typing import Optional
import pandas as pd


def filter_stocks_by_profile(
    df: pd.DataFrame,
    risk: Optional[str] = None,      # "low" or "high"
    horizon: Optional[str] = None,   # "long" or "short"
    sector: Optional[str] = None,
    top_n: int = 10,
) -> pd.DataFrame:
    """
    Filter and rank stocks based on user profile:
    - risk: "low" or "high"
    - horizon: "long" or "short"
    - sector: optional sector filter
    - top_n: number of rows to return

    Uses columns from combined_quantamental_hybrid_with_factors_and_backtest.csv.
    """

    f = df.copy()

    # Always exclude bearish names
    f = f[f["H_Score Recommendation"] != "Avoid / Bearish"]

    # Sector filter
    if sector:
        f = f[f["sector"] == sector]

    rec = f["H_Score Recommendation"]

    # Default scenario: if user did not specify anything
    if risk is None and horizon is None:
        risk = "low"
        horizon = "long"

    # Handle invalid risk/horizon values defensively
    if risk not in {"low", "high"} and risk is not None:
        raise ValueError(f"Invalid risk value: {risk}")
    if horizon not in {"long", "short"} and horizon is not None:
        raise ValueError(f"Invalid horizon value: {horizon}")

    # === Scenario 1: Long-Term + Low-Risk (LT–LR) ===
    if risk == "low" and horizon == "long":
        f = f[
            rec.isin(["Long-Term Buy (Fundamental)", "Balanced Buy / Hold"])
            & (f["cagr"] >= 0.17)
            & (f["sharpe_1m_annual"] >= 0.8)
            & (f["vol_1m"] <= 0.075)
            & (f["max_drawdown"] >= -0.30)
            & (f["hit_rate_pos"] >= 0.55)
        ]
        sort_cols = ["cagr", "sharpe_1m_annual", "vol_1m"]
        ascending = [False, False, True]

    # === Scenario 2: Long-Term + High-Risk (LT–HR) ===
    elif risk == "high" and horizon == "long":
        f = f[
            rec.isin(
                [
                    "Long-Term Buy (Fundamental)",
                    "Balanced Buy / Hold",
                    "Short-Term Buy (Momentum)",
                ]
            )
            & (f["cagr"] >= 0.24)
            & (f["sharpe_1m_annual"] >= 0.5)
            & (f["max_drawdown"] >= -0.40)
            & (f["Hybrid_CS_Pct"] >= 0.70)
        ]
        sort_cols = ["cagr", "Hybrid_Score", "vol_1m"]
        ascending = [False, False, True]

    # === Scenario 3: Short-Term + Low-Risk (ST–LR) ===
    elif risk == "low" and horizon == "short":
        f = f[
            rec.isin(["Short-Term Buy (Momentum)", "Balanced Buy / Hold"])
            & (f["avg_fwd_1m_ret"] >= 0.016)
            & (f["vol_1m"] <= 0.089)
            & (f["max_drawdown"] >= -0.30)
            & (f["hit_rate_pos"] >= 0.60)
        ]
        sort_cols = ["avg_fwd_1m_ret", "hit_rate_pos", "vol_1m"]
        ascending = [False, False, True]

    # === Scenario 4: Short-Term + High-Risk (ST–HR) ===
    elif risk == "high" and horizon == "short":
        f = f[
            (rec == "Short-Term Buy (Momentum)")
            & (f["avg_fwd_1m_ret"] >= 0.021)
            & (f["Hybrid_CS_Pct"] >= 0.80)
            & (f["vol_1m"] >= 0.075)
            & (f["max_drawdown"] >= -0.45)
            & (f["sharpe_1m_annual"] > 0.0)
        ]
        sort_cols = ["avg_fwd_1m_ret", "Hybrid_CS_Pct", "vol_1m"]
        ascending = [False, False, False]  # we like high vol here

    else:
        # Invalid combination or not covered; return empty
        return f.head(0)

    if f.empty:
        return f  # nothing matched

    f = f.sort_values(sort_cols, ascending=ascending)

    # Limit to top_n and return
    return f.head(top_n)

FastAPI /stocks/filter endpoint

Assume:

Backend package: src/backend

You have the CSV in data/combined_quantamental_hybrid_with_factors_and_backtest.csv

You place filter_stocks_by_profile in src/backend/services/filter_service.py

In [ ]:
# src/backend/services/filter_service.py

import pandas as pd
from functools import lru_cache
from typing import Optional

from .filtering_logic import filter_stocks_by_profile  # or keep function here


@lru_cache(maxsize=1)
def load_quant_df() -> pd.DataFrame:
    """
    Load the combined quant file once per process.
    Adjust path as needed to where the CSV is stored in the container.
    """
    df = pd.read_csv(
        "data/combined_quantamental_hybrid_with_factors_and_backtest.csv"
    )
    return df


def get_filtered_stocks(
    risk: Optional[str],
    horizon: Optional[str],
    sector: Optional[str],
    top_n: int = 10,
):
    df = load_quant_df()
    filtered = filter_stocks_by_profile(
        df=df,
        risk=risk,
        horizon=horizon,
        sector=sector,
        top_n=top_n,
    )

    # Rename columns with spaces to API-friendly names
    rename_cols = {
        "H_Score Recommendation": "H_Score_Recommendation",
    }
    filtered = filtered.rename(columns=rename_cols)

    # Optionally, select a subset of columns for the API response
    cols = [
        "symbol",
        "sector",
        "H_Score_Recommendation",
        "Hybrid_Score",
        "Hybrid_CS_Pct",
        "cagr",
        "sharpe_1m_annual",
        "vol_1m",
        "max_drawdown",
        "hit_rate_pos",
        "avg_fwd_1m_ret",
    ]
    existing_cols = [c for c in cols if c in filtered.columns]

    return filtered[existing_cols].to_dict(orient="records")

In [ ]:
# src/backend/routes/filter.py

from typing import Optional, Literal

from fastapi import APIRouter, Query, HTTPException

from ..services.filter_service import get_filtered_stocks

router = APIRouter(
    prefix="/stocks",
    tags=["stocks"],
)


@router.get("/filter")
def filter_stocks(
    risk: Optional[Literal["low", "high"]] = Query(
        default=None,
        description="Risk preference: 'low' or 'high'. If omitted, defaults to low when horizon is also omitted.",
    ),
    horizon: Optional[Literal["long", "short"]] = Query(
        default=None,
        description="Investment horizon: 'long' or 'short'. If omitted with risk, defaults to long-term low-risk.",
    ),
    sector: Optional[str] = Query(
        default=None,
        description="Optional sector filter (e.g. 'Technology', 'Healthcare').",
    ),
    top_n: int = Query(
        default=10,
        ge=1,
        le=100,
        description="Max number of stocks to return.",
    ),
):
    """
    Filter stocks based on user risk, horizon, and sector preference.

    Examples:
    - /stocks/filter?risk=low&horizon=long
    - /stocks/filter?risk=high&horizon=short&sector=Technology
    - /stocks/filter          -> defaults to scenario: Long-Term + Low-Risk
    """

    try:
        results = get_filtered_stocks(
            risk=risk,
            horizon=horizon,
            sector=sector,
            top_n=top_n,
        )
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))

    return {
        "risk": risk,
        "horizon": horizon,
        "sector": sector,
        "count": len(results),
        "results": results,
    }


In [ ]:
# src/backend/main.py

from fastapi import FastAPI
from .routes import filter as filter_routes
# (plus any existing routes you already have)

app = FastAPI(
    title="LLM-Quant Stock Screener API",
    version="0.1.0",
)

app.include_router(filter_routes.router)


@app.get("/health")
def health():
    return {"status": "ok"}


**✅ 1. Frontend API Client for /stocks/filter (React + TypeScript)**

This goes inside your React project (e.g., frontend/src/api/client.ts).

It supports:

risk

horizon

sector

top_n

returns typed results

handles Cloud Run base URL from .env

In [ ]:
// src/api/client.ts

export interface FilteredStock {
  symbol: string;
  sector: string;
  H_Score_Recommendation: string;
  Hybrid_Score: number;
  Hybrid_CS_Pct: number;
  cagr: number;
  sharpe_1m_annual: number;
  vol_1m: number;
  max_drawdown: number;
  hit_rate_pos: number;
  avg_fwd_1m_ret: number;
}

export interface FilterResponse {
  risk: string | null;
  horizon: string | null;
  sector: string | null;
  count: number;
  results: FilteredStock[];
}

const API_BASE_URL = import.meta.env.VITE_API_URL; 
// Example: https://llm-quant-backend-xxxxx-uc.a.run.app

export async function filterStocks(params: {
  risk?: "low" | "high";
  horizon?: "long" | "short";
  sector?: string | null;
  top_n?: number;
}): Promise<FilterResponse> {
  const url = new URL(`${API_BASE_URL}/stocks/filter`);

  if (params.risk) url.searchParams.append("risk", params.risk);
  if (params.horizon) url.searchParams.append("horizon", params.horizon);
  if (params.sector) url.searchParams.append("sector", params.sector);
  if (params.top_n) url.searchParams.append("top_n", String(params.top_n));

  const res = await fetch(url.toString());
  if (!res.ok) throw new Error(`API Error: ${res.status}`);

  return res.json() as Promise<FilterResponse>;
}

In [ ]:
##Example usage in your UI (React)

##Button click triggers filtering:

import { filterStocks } from "../api/client";

const loadStocks = async () => {
  const data = await filterStocks({
    risk: "low",
    horizon: "long",
    sector: "Healthcare",
    top_n: 10,
  });

  console.log(data.results);
}

#This returns the fully filtered list ready for rendering in a table or chart.

2. LLM Explanation Templates (Scenario-Aware)

These templates allow your LLM (Vertex AI or your RAG service) to produce human-friendly, personalized explanations for the recommended stocks.

Each template is scenario-specific and uses stock metrics.
🎯 BASE TEMPLATE FOR ALL SCENARIOS

Use this for your /llm/explain endpoint.

In [ ]:
llm_prompt_template.txt
You are an AI financial analyst specializing in Quantamental investing.

Your goal is to explain why the following stocks were recommended based on:
- User’s risk preference ({{risk}})
- Investment horizon ({{horizon}})
- Optional sector filter ({{sector}})
- The model’s metrics: Hybrid Score, Sharpe, volatility, drawdown, CAGR,
  hit-rate, forward returns, and H_Score Recommendation.

You must:
1. Explain the user's scenario type.
2. Explain the selection logic clearly.
3. Provide a per-stock breakdown with metric highlights.
4. Summarize why these stocks fit the user's strategy.
5. Explain risk factors and what to watch out for.

Here is the scenario classification:

Long-Term Low-Risk (LT–LR):
- stable CAGR, high Sharpe, low volatility, shallow drawdowns
Long-Term High-Risk (LT–HR):
- high CAGR, moderate Sharpe, deeper drawdowns acceptable
Short-Term Low-Risk (ST–LR):
- stable momentum, positive forward returns, controlled vol
Short-Term High-Risk (ST–HR):
- strong momentum, high volatility, high Hybrid_CS_Pct

User Profile:
- risk = {{risk}}
- horizon = {{horizon}}
- sector = {{sector}}

Recommended Stocks (JSON):
{{stocks_json}}

Generate a clear narrative explanation.
Use bullet points and highlight metrics.
Avoid giving financial advice—focus on interpretation.

Output structure:
1. Scenario Explanation
2. Why This Scenario Selects These Types of Stocks
3. Stock-by-Stock Justification
4. Strengths & Risks
5. Final Summary

In [ ]:
Scenario-specific additions

You prepend these to the template depending on the scenario.

🟩 LT–LR (Long-Term + Low-Risk)
This user prefers a long-term, low-risk investment style. 
The goal is stable compounding with limited volatility.

The system emphasizes:
- High Sharpe ratios (risk-adjusted return)
- Low volatility
- Strong long-term CAGR
- Controlled drawdowns
- “Long-Term Buy (Fundamental)” or “Balanced Buy / Hold” signals

In [ ]:
LT–HR (Long-Term + High-Risk)
This user prefers long-term growth and can tolerate higher volatility.

The system emphasizes:
- High multi-year CAGR
- Strong hybrid scores
- Momentum + fundamentals
- Higher acceptable drawdowns

In [ ]:
ST–LR (Short-Term + Low-Risk)
This user prefers short-term opportunities but wants controlled risk.

The system emphasizes:
- Strong 1-month forward returns
- High hit-rate of positive months
- Low-to-moderate monthly volatility
- Momentum signals with stability

In [ ]:
ST–HR (Short-Term + High-Risk)
This user accepts volatility and seeks high-momentum short-term trades.

The system emphasizes:
- Top-tier short-term momentum
- High Hybrid_CS_Pct
- High forward return expectancy
- Acceptable but higher drawdown and volatility

🎤 Example LLM-Generated Output (demonstration)

Given: LT–LR + Healthcare sector
Stocks: HCA, UHS

The LLM will produce something like:

1. Scenario Explanation

You selected a long-term, low-risk strategy.
This prioritizes stable compounders that generate strong returns with lower volatility and shallow drawdowns.

2. Why These Stocks Fit

High Sharpe ratios → consistent risk-adjusted performance

Low volatility → stable month-to-month behavior

Strong CAGR → good long-term compounding potential

“Balanced Buy / Hold” or “Long-Term Buy” signals

3. Stock Breakdown
HCA

Sharpe: 1.12 (excellent)

Volatility: 0.07 (below market median)

CAGR: 18.4%

Hit Rate: 61%

H_Score: Balanced Buy / Hold

This makes HCA a strong long-term compounder with controlled risk.

UHS

Sharpe: 0.93

Volatility: 0.066 (low)

CAGR: 16.1%

H_Score: Long-Term Buy (Fundamental)

This stock demonstrates strong fundamentals and stable growth.

4. Strengths & Risks

Strengths:

Defensive healthcare sector

Stable earnings

Lower volatility
Risks:

Healthcare regulatory cycles

Slower upside vs high-risk growth stocks

5. Summary

These recommendations match your long-term, low-risk profile by emphasizing stability, quality, and risk-managed compounding.